# Chapter 2: Working with Text Data

Packages that are being used in this notebook:

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.15.0
tiktoken version: 0.14.0


## 2.2 Tokenizing text

- In this section, we tokenize text, which means breaking text into smaller units, such as individual words and punctuation characters

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

Loading the data from the downloaded text file

In [2]:
# This line of code reads the entire content of a text file into a Python variable.
# with opens the file in read mode and as f assigns the file object to the variable f.
# 'r' means read mode, and 'utf-8' is the encoding used to read the file.
# f.read() reads the entire content of the file and stores it in the variable text.

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
type(raw_text)

str

- The goal is to tokenize and embed this text for an LLM
- Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
- The following regular expression will split on whitespaces

In [3]:
import re

text = "Hello, world. This, is a test."

In [4]:
results = re.split(r'(\s)', text)   # this regular expression r'(\s)' will break the text on white spaces
print(results)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [7]:
text.split()    # this is built in method in python but it removes whitespaces

['Hello,', 'world.', 'This,', 'is', 'a', 'test.']

- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [11]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- As we can see, this creates empty strings, let's remove them

In [13]:
# Strip whitespace from each item and then filter out any empty strings.

results = [item for item in result if item.strip()]
print(results) 

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [19]:
# In Python, an empty string '' is considered Falsy (evaluates to False), while any string with at least one character left (e.g., 'Hello' or ',') is Truthy (evaluates to True).

a = ""
print(bool(a.strip()))  # Output: False

False


- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on

In [14]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [15]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Let's calculate the total number of tokens

In [20]:
print(len(preprocessed))

4690


## 2.3 Converting tokens into token IDs

- Next, we convert the text tokens into token IDs that we can process via embedding layers later

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- From these tokens, we can now build a vocabulary that consists of all the unique tokens

In [22]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [23]:
vocab = {token:integer for integer, token in enumerate(all_words)}

- Below are the first 50 entries in this vocabulary:

In [26]:
# WITH .items() -> prints key-value pairs
# WITHOUT .items() -> prints only keys
for index, item in enumerate(vocab.items()):
    print(item)
    if index >= 50:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Below, we illustrate the tokenization of a short sample text using a small vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

In [37]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        # removing empty space
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        # accessing dictionary key-value
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


- The `encode` function turns text into token IDs
- The `decode` function turns token IDs back into text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- We can use the tokenizer to encode (that is, tokenize) texts into integers
- These integers can then be embedded (later) as input of/for the LLM

In [38]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- We can decode the integers back into text

In [39]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [40]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

- It's useful to add some "special" tokens for unknown words and to denote the end of a text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Some tokenizers use special tokens to help the LLM with additional context
- Some of these special tokens are
  - `[BOS]` (beginning of sequence) marks the beginning of text
  - `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
  - `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
- `[UNK]` to represent words that are not included in the vocabulary

- Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity
- The `<|endoftext|>` is analogous to the `[EOS]` token mentioned above
- GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)
- GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding (BPE) tokenizer, which breaks down words into subword units which we will discuss in a later section

- We use the `<|endoftext|>` tokens between two independent sources of text:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- Let's see what happens if we tokenize the following text:

In [41]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- The above produces an error because the word "Hello" is not contained in the vocabulary
- To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words
- Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)

In [42]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [43]:
len(vocab.items())

1132

In [44]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token

In [45]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Let's try to tokenize text with the modified tokenizer:

In [49]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))   #adding EOD manually

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [50]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [51]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 BytePair encoding

- GPT-2 used BytePair encoding (BPE) as its tokenizer
- it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words
- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges
- The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- In this chapter, we are using the BPE tokenizer from OpenAI's open-source [tiktoken](https://github.com/openai/tiktoken) library, which implements its core algorithms in Rust to improve computational performance
- I created a notebook in the [./bytepair_encoder](../02_bonus_bytepair-encoder) that compares these two implementations side-by-side (tiktoken was about 5x faster on the sample text)

In [52]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.14.0


In [53]:
tokenizer = tiktoken.get_encoding("gpt2")

In [54]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [55]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizers break down unknown words into subwords and individual characters:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 Data sampling with a sliding window

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [56]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- For each text chunk, we want the inputs and targets
- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right

In [57]:
enc_sample = enc_text[50:]

In [58]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- One by one, the prediction would look like as follows:

In [59]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [60]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- We will take care of the next-word prediction in a later chapter after we covered the attention mechanism
- For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one

In [61]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.15.0


- We use a sliding window approach, changing the position by +1:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- Create dataset and dataloader that extract chunks from the input text dataset


max_length defines the fixed sequence length (number of tokens) of each training example sliced from your full text. It also acts as the model's context window size during training—the maximum number of tokens the GPT model sees at once when predicting the next token.

In [62]:

from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length

        for i in range(0, len(token_ids), stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+stride:i+max_length+stride]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
            return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]



In [63]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:

In [64]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [65]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [66]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- An example using stride equal to the context length (here: 4) as shown below:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- We can also create batched outputs
- Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting

In [67]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11],
        [  287,   262,  6001,   286]])


## 2.7 Creating token embeddings

- The data is already almost ready for an LLM
- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):

In [68]:
input_ids = torch.tensor([2, 3, 5, 1])

- For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:

In [69]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- This would result in a 6x3 weight matrix:

In [72]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [71]:
print(embedding_layer.weight.shape)

torch.Size([6, 3])


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer, which is described in the supplementary code in [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)
- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

An embedding layer is mathematically identical to multiplying a **one-hot encoded vector** by a **weight matrix**, except it skips all the wasted multiplications by zero and simply looks up the row directly.

### 1. Why They Are Mathematically Identical

Suppose your vocabulary has 4 words (`vocab_size = 4`) and each word gets a 3-number embedding (`embedding_dim = 3`). Your embedding weight matrix $W$ has shape $4 \times 3$:

$$W = \begin{bmatrix} 0.1 & 0.2 & 0.3 \\ 0.4 & 0.5 & 0.6 \\ 0.7 & 0.8 & 0.9 \\ 1.0 & 1.1 & 1.2 \end{bmatrix}$$

Now look at what happens when you want the embedding for **Token ID `2`** (the 3rd row):

* **Method A: One-Hot Encoding + Matrix Multiplication**
  1. Turn token ID `2` into a one-hot vector of length 4: $\begin{bmatrix} 0 & 0 & 1 & 0 \end{bmatrix}$
  2. Multiply that vector by $W$:
     $$\begin{bmatrix} 0 & 0 & 1 & 0 \end{bmatrix} \times \begin{bmatrix} 0.1 & 0.2 & 0.3 \\ 0.4 & 0.5 & 0.6 \\ 0.7 & 0.8 & 0.9 \\ 1.0 & 1.1 & 1.2 \end{bmatrix} = \begin{bmatrix} 0.7 & 0.8 & 0.9 \end{bmatrix}$$
  3. Notice what happened: the zeros wiped out rows `0`, `1`, and `3`, and the `1` simply copied **row `2`**.

* **Method B: `nn.Embedding` Lookup**
  Instead of creating a huge vector full of zeros (imagine 50,257 zeros for GPT-2) and multiplying everything by zero, `nn.Embedding` jumps straight to **row `2`** of $W$ and returns $\begin{bmatrix} 0.7 & 0.8 & 0.9 \end{bmatrix}$.

| Feature | One-Hot + `nn.Linear` | `nn.Embedding` |
| :--- | :--- | :--- |
| **Input** | One-hot vector `[0, 0, 1, 0]` | Integer token ID `2` |
| **Operation** | Full matrix multiplication | Direct row index lookup (`W[2]`) |
| **Output** | `[0.7, 0.8, 0.9]` | `[0.7, 0.8, 0.9]` |
| **Speed / Memory** | Slow and memory-heavy for large vocabularies | Fast and memory-efficient |

---

### 2. Why It Can Be Trained via Backpropagation

Because `nn.Embedding` produces the **exact same output** as a standard `nn.Linear` layer without a bias term, PyTorch treats the embedding table $W$ as a normal learnable parameter matrix:

1. **Forward pass:** PyTorch fetches row `2` (`W[2]`) and remembers which token IDs were looked up.
2. **Backward pass (Backpropagation):** When the loss is calculated, gradients flow backward only into the specific rows that were looked up (e.g., row `2`), updating those numbers so that words used in similar contexts gradually move closer together in embedding space.

In [74]:
import torch
import torch.nn as nn

vocab_size, emb_dim = 4, 3
token_id = torch.tensor([2])

# 1. Embedding layer
emb = nn.Embedding(vocab_size, emb_dim)

# 2. Equivalent Linear layer (must transpose weights and disable bias)
lin = nn.Linear(vocab_size, emb_dim, bias=False)
lin.weight = nn.Parameter(emb.weight.T)

one_hot = torch.nn.functional.one_hot(token_id, num_classes=vocab_size).float()

print(emb(token_id))   # Direct lookup
print(lin(one_hot))    # Exact same output via matrix multiplication

tensor([[-0.9178,  0.9045, -2.0975]], grad_fn=<EmbeddingBackward0>)
tensor([[-0.9178,  0.9045, -2.0975]], grad_fn=<MmBackward0>)


The math behind an embedding layer involves **indexing**, **vector projection**, and **sparse gradient updates**. 

Instead of complex equations, an embedding layer relies on a single parameter matrix $W$ of shape $(V, d)$, where:
* $V$ = vocabulary size (number of unique tokens)
* $d$ = embedding dimension (size of each vector)

---

### 1. The Forward Pass: Matrix Multiplication vs. Row Lookup

Suppose we have a sequence of $T$ token IDs: $(x_1, x_2, \dots, x_T)$, where each $x_t \in \{0, 1, \dots, V-1\}$.

#### Form A: As a Linear Projection (One-Hot Matrix Multiplication)
Each token ID $x_t$ is represented as a **one-hot row vector** $\mathbf{e}_{x_t} \in \mathbb{R}^{1 \times V}$ with a $1$ at index $x_t$ and $0$ everywhere else:

$$\mathbf{e}_{x_t} = \begin{bmatrix} 0 & \dots & 1 & \dots & 0 \end{bmatrix}$$

To get the embedding vector $\mathbf{h}_t \in \mathbb{R}^{1 \times d}$, we multiply this one-hot vector by the embedding matrix $W \in \mathbb{R}^{V \times d}$:

$$\mathbf{h}_t = \mathbf{e}_{x_t} W$$

Writing out the dot product for the $j$-th element of $\mathbf{h}_t$:

$$h_{t,j} = \sum_{k=0}^{V-1} (\mathbf{e}_{x_t})_k \cdot W_{k,j}$$

Because $(\mathbf{e}_{x_t})_k = 0$ for every $k \neq x_t$, every term in the sum disappears except $k = x_t$:

$$h_{t,j} = 1 \cdot W_{x_t, j} = W_{x_t, j}$$

#### Form B: As Direct Indexing (What Actually Runs in Code)
Because the sum collapses to a single entry, the entire forward pass simplifies to indexing row $x_t$ of matrix $W$:

$$\mathbf{h}_t = W[x_t, :]$$

For a full sequence of $T$ tokens, stacking the outputs produces a matrix $H \in \mathbb{R}^{T \times d}$.

---

### 2. The Backward Pass: Sparse Gradient Updates

During training, a loss function $\mathcal{L}$ measures how wrong the model's prediction was. Using backpropagation, the model receives the gradient of the loss with respect to the embedding output, $\frac{\partial \mathcal{L}}{\partial \mathbf{h}_t} \in \mathbb{R}^{1 \times d}$.

To update the weight matrix $W$, we need $\frac{\partial \mathcal{L}}{\partial W}$. Applying the chain rule to $\mathbf{h}_t = \mathbf{e}_{x_t} W$:

$$\frac{\partial \mathcal{L}}{\partial W} = \sum_{t=1}^{T} \mathbf{e}_{x_t}^\top \frac{\partial \mathcal{L}}{\partial \mathbf{h}_t}$$

Because $\mathbf{e}_{x_t}^\top$ is a column vector with $1$ only at row $x_t$ and $0$ elsewhere, the outer product $\mathbf{e}_{x_t}^\top \frac{\partial \mathcal{L}}{\partial \mathbf{h}_t}$ creates a $V \times d$ matrix that is **all zeros except at row $x_t$**:

$$\frac{\partial \mathcal{L}}{\partial W_{k, :}} = \sum_{t \,:\, x_t = k} \frac{\partial \mathcal{L}}{\partial \mathbf{h}_t}$$

**What this means:**
* If token $k$ appeared **0 times** in the batch, its gradient is $\mathbf{0}$ and its row in $W$ stays completely unchanged.
* If token $k$ appeared **once**, row $k$ gets updated by $-\eta \frac{\partial \mathcal{L}}{\partial \mathbf{h}_t}$ (where $\eta$ is the learning rate).
* If token $k$ appeared **multiple times** in the same batch, its gradients from each position are summed together before updating row $k$.

---

### 3. Concrete Numerical Walkthrough

Suppose $V = 3$, $d = 2$, learning rate $\eta = 0.1$, and initial weights are:

$$W = \begin{bmatrix} 0.2 & 0.5 \\ -0.3 & 0.8 \\ 0.4 & -0.1 \end{bmatrix}$$

1. **Input:** Token ID $x = 1$ (second row).
2. **Forward pass

- To convert a token with id 3 into a 3-dimensional vector, we do the following:

In [73]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Note that the above is the 4th row in the `embedding_layer` weight matrix
- To embed all four `input_ids` values above, we do

In [75]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- An embedding layer is essentially a look-up operation:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- You may be interested in the bonus content comparing embedding layers with regular linear layers

## 2.8 Encoding word positions

- Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

Imagine you're giving words to a language model like this:

"The cat sat on the mat"

The model turns each word into a vector using the embedding layer, like:

"The" → [0.12, -0.45, ...]
"cat" → [0.33, 0.88, ...]

BUT — here’s the catch:

The same word always gets the same vector, no matter where it is in the sentence.

So if you give:

"The cat sat" → [the, cat, sat]
"Sat cat the" → [sat, cat, the]
They all get the same vectors, even though they mean different things in different spots.
But it doesn’t know that the first one makes sense and the second one doesn’t!

Why? Because...
The self-attention mechanism in transformers (the heart of GPT/BERT/LLAMA) looks at all words at once, but without any built-in idea of their position.|

The Solution: Add Position Info
To fix that, we add position encodings — basically a way to tell the model:

"Hey, this word is the 1st in the sentence, this one is 2nd, this one is 3rd..."

So now:

"The" at position 0 → [token_vector] + [position_vector_0]  
"cat" at position 1 → [token_vector] + [position_vector_1]

Now the same word ("the") will look slightly different depending on where it is in the sentence. That helps the model understand order.


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- The BytePair encoder has a vocabulary size of 50,257:
- Suppose we want to encode the input tokens into a 256-dimensional vector representation:

In [76]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:

In [77]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [78]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [79]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


- GPT-2 uses absolute position embeddings, so we just create another embedding layer:

In [81]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to
# print(pos_embedding_layer.weight)

In [83]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(pos_embeddings)

torch.Size([4, 256])
tensor([[-0.1224, -0.4060, -0.6279,  ..., -0.5005, -0.1670, -0.4342],
        [-1.6298, -0.5170,  1.4878,  ..., -0.0637, -1.7915,  1.2324],
        [-1.7268,  1.4151, -0.7310,  ...,  1.2285,  0.3184,  2.0517],
        [-3.0392,  1.0096,  0.0503,  ...,  1.8649,  1.6372, -2.1538]],
       grad_fn=<EmbeddingBackward0>)


- To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:

In [84]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


- In the initial phase of the input processing workflow, the input text is segmented into separate tokens
- Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">